# Machine Sound Fault Detection - Convolutional Autoencoder (CAE) (MIMII)

This notebook finds **faulty machines by their sound**. It trains on **healthy** sound only.

**How it works**
1. The model learns to copy healthy sound.
2. A faulty sound is copied badly. A **big copy error = possible fault**.
3. The alarm threshold is the **95th percentile of healthy validation scores**. Faulty clips are never used to train or to set the threshold.

**What you need to do**
1. Add the MIMII dataset (**Add Input**).
2. Turn on the GPU (**Settings > Accelerator > GPU**).
3. Click **Run All**.

**Default = full run:** 4 machines x 4 IDs (id_00, id_02, id_04, id_06) x 6 dB noise = 16 runs.
**Want a fast check first?** Set `QUICK = True` in Step 2 (2 IDs, 150 clips per class).

The model is a **2D convolutional autoencoder** as in `CLAUDE.md`: 4 conv blocks down, a small 32-number middle layer, 4 blocks up, no skip connections. It trains slower than the dense one.

## Words used in this notebook

| Word | Meaning |
|---|---|
| **clip** | One 10-second sound file |
| **window** | A 2-second piece of a clip |
| **healthy / faulty** | Machine works well / machine has a problem |
| **score** | The copy error of a clip. Higher = more likely faulty |
| **AUC** | How well the score separates healthy from faulty. **1.0 = perfect, 0.5 = random guess** |
| **pAUC** | Same, but only when false alarms are under 10% |
| **recall** | Share of faulty clips that we catch |
| **false alarm** | Share of healthy clips we wrongly call faulty |
| **balanced accuracy** | Average of recall and "healthy kept". Fair when classes are not equal |
| **threshold** | The score above which we say "faulty" |

## Step 1 - Import tools

In [ ]:
import glob, gc, re, sys, time, os, json, subprocess, importlib.util, shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from joblib import Parallel, delayed
from sklearn.metrics import (roc_auc_score, roc_curve, f1_score, confusion_matrix, ConfusionMatrixDisplay)
from IPython.display import display

for name in ("librosa", "soundfile"):
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", name])
import librosa
import soundfile as sf

## Step 2 - Settings

- `QUICK = False` is the full run (16 runs). `QUICK = True` is a fast check.
- `SPLIT_MODE`: `"random"` picks the healthy clips for train / validation / test at random. `"blocked"` cuts them in file-name order (first 70% train, next 10% validation, last 20% test). My guess is that file order follows recording time, so `"blocked"` is a **stricter** test. I have not checked this.

In [ ]:
# ---- Where things are ----
DATA_ROOT = "/kaggle/input"      # the audio files are somewhere inside this folder
WORK_DIR  = "/kaggle/working"    # results are saved here
TAG       = "cae"

# ---- What to run ----
QUICK     = False                # True = fast check (2 IDs, 150 clips per class)
MACHINES  = ["fan", "pump", "slider", "valve"]
IDS       = ["id_00", "id_02"] if QUICK else ["id_00", "id_02", "id_04", "id_06"]
LIMIT     = 150 if QUICK else None   # max clips per class per run (random sample)
SNR       = 6                    # noise level: 6, 0 or -6
SPLIT_MODE = "random"            # "random" or "blocked" (see above)
MAX_HOURS = 11                   # stop starting new runs after this many hours

# ---- Sound settings ----
SAMPLE_RATE = 16000
N_FFT, HOP, N_MELS = 1024, 512, 128
MIC     = 0                      # the dataset has 8 microphones. We use the first one.
WINDOW  = 32768                  # samples per window (about 2 seconds)
STEP    = 16384                  # the next window starts 1 second later
FRAMES  = 64                     # columns kept in each picture

# ---- Model settings ----
LATENT    = 32                   # size of the small middle layer (32 as in CLAUDE.md)
BATCH     = 64
LEARNING_RATE = 1e-3
EPOCHS    = 60                   # at most 60 epochs (early stopping usually stops sooner)
AUGMENT   = True                 # random volume change of +/- GAIN_DB on training data only
GAIN_DB   = 2.0
NORMALIZE = "minmax"             # "minmax" = as in CLAUDE.md (0..1, sigmoid output)
                                 # "zscore" = like the dense model (each mel row standardised, linear output)
THRESHOLD_PERCENT = 95           # threshold = 95th percentile of healthy validation scores
SEED      = 42

MACHINE_COLOR = {"fan": "#1f77b4", "pump": "#ff7f0e", "slider": "#2ca02c", "valve": "#d62728"}
FIG_DIR = WORK_DIR + "/figures_" + TAG
os.makedirs(FIG_DIR, exist_ok=True)

def show(name):
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/{name}.png", dpi=110, bbox_inches="tight")
    plt.show()

## Step 3 - Check the GPU

You should see one GPU. If the list is empty, turn on the GPU in **Settings > Accelerator**.

In [ ]:
tf.keras.utils.set_random_seed(SEED)
gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
print("TensorFlow", tf.__version__)
print("GPUs found:", [g.name for g in gpus])

## Step 4 - Find the audio files

In [ ]:
def read_path(path):
    parts = path.split("/")[::-1]          # start from the file name
    label   = next((p for p in parts if p in ("normal", "abnormal")), None)
    machine = next((p for p in parts if p in ("fan", "pump", "slider", "valve")), None)
    mid     = next((p for p in parts if re.fullmatch(r"id_\d+", p)), None)
    snr = None
    for p in parts:
        m = re.match(r"(-?\d+)\s*_?dB", p)
        if m:
            snr = int(m.group(1))
            break
    return label, machine, mid, snr

rows = []
for path in glob.glob(DATA_ROOT + "/**/*.wav", recursive=True):
    label, machine, mid, snr = read_path(path)
    if label and machine and mid:
        rows.append([path, machine, mid, label, snr])

df = pd.DataFrame(rows, columns=["path", "machine", "id", "label", "snr"])
assert len(df) > 0, "No audio found. Did you add the dataset to this notebook (Add Input)?"
print(len(df), "clips found")

Number of clips per machine, ID, noise level and label:

In [ ]:
counts = df.groupby(["machine", "id", "snr", "label"], dropna=False).size().unstack("label", fill_value=0)
display(counts)

## Step 5 - Choose the runs

One run = one machine + one ID. If `LIMIT` is set, we take a **random** sample of that many clips per class (the same sample every time).

In [ ]:
USE_SNR = df["snr"].notna().any()     # some datasets have no noise level in the folder name

def get_files(machine, mid):
    rows = df[(df["machine"] == machine) & (df["id"] == mid)]
    if USE_SNR:
        rows = rows[rows["snr"] == SNR]
    healthy = sorted(rows[rows["label"] == "normal"]["path"])
    faulty  = sorted(rows[rows["label"] == "abnormal"]["path"])
    if LIMIT:
        rng = np.random.default_rng(SEED)
        if len(healthy) > LIMIT: healthy = sorted(rng.choice(healthy, LIMIT, replace=False).tolist())
        if len(faulty)  > LIMIT: faulty  = sorted(rng.choice(faulty,  LIMIT, replace=False).tolist())
    return healthy, faulty

plan = []
for machine in MACHINES:
    for mid in IDS:
        healthy, faulty = get_files(machine, mid)
        if len(healthy) >= 30 and len(faulty) >= 5:
            plan.append((machine, mid))
        else:
            print("skip", machine, mid, "- not enough clips")

print(len(plan), "runs planned:")
print(plan)

## Step 6 - Turn sound into a picture

A **log-mel picture**: 128 rows (pitch) x 64 columns (time) for each 2-second window.

In [ ]:
def load_audio(path):
    audio, rate = sf.read(path, dtype="float32", always_2d=True)   # shape: (samples, microphones)
    audio = audio[:, min(MIC, audio.shape[1] - 1)]                 # keep one microphone
    if rate != SAMPLE_RATE:
        audio = librosa.resample(audio, orig_sr=rate, target_sr=SAMPLE_RATE)
    return audio

def cut_windows(audio):
    if len(audio) < WINDOW:
        audio = np.pad(audio, (0, WINDOW - len(audio)))
    starts = range(0, len(audio) - WINDOW + 1, STEP)
    return [audio[s:s + WINDOW] for s in starts]

def to_logmel(window):
    mel = librosa.feature.melspectrogram(y=window, sr=SAMPLE_RATE, n_fft=N_FFT,
                                         hop_length=HOP, n_mels=N_MELS)
    db = librosa.power_to_db(mel, ref=1.0)
    return db[:, :FRAMES]                                          # shape: (128, 64)

Example: one healthy and one faulty window.

In [ ]:
machine, mid = plan[0]
healthy, faulty = get_files(machine, mid)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for ax, path, name in zip(axs, [healthy[0], faulty[0]], ["healthy", "faulty"]):
    ax.imshow(to_logmel(load_audio(path)[:WINDOW]), origin="lower", aspect="auto", cmap="magma")
    ax.set(title=f"{machine} {mid} - {name}", xlabel="time", ylabel="mel band")
show("example_spectrograms")

## Step 7 - Load the clips

In [ ]:
def clip_to_pictures(path):
    windows = cut_windows(load_audio(path))
    return np.array([to_logmel(w) for w in windows], dtype=np.float32)

def load_clips(files):
    pieces = Parallel(n_jobs=-1)(delayed(clip_to_pictures)(f) for f in files)
    pictures = np.concatenate(pieces)                                          # all windows together
    clip_id = np.concatenate([np.full(len(p), i) for i, p in enumerate(pieces)])  # which clip each window came from
    return pictures, clip_id

## Step 8 - Split the healthy clips

70% train, 10% validation, 20% test. **All faulty clips go to the test set.**
The split is by clip, so no clip is in two sets.

In [ ]:
def split_healthy(healthy_files):
    files = sorted(healthy_files)
    n = len(files)
    order = np.random.default_rng(SEED).permutation(n) if SPLIT_MODE == "random" else np.arange(n)
    n_train, n_val = int(0.7 * n), int(0.1 * n)
    pick = lambda idx: [files[i] for i in idx]
    return pick(order[:n_train]), pick(order[n_train:n_train + n_val]), pick(order[n_train + n_val:])

## Step 9 - Scale the pictures

Two options (`NORMALIZE` in Step 2). The scaling numbers always come from the **train set only**.
- `"minmax"` (as in `CLAUDE.md`): one `lo` and one `hi` for the whole picture, values 0..1. The model output is a sigmoid (0..1).
- `"zscore"` (like the dense model): each mel row is standardised with its own mean and spread. The model output is linear.

During training we add a random volume change of +/- 2 dB (as in `CLAUDE.md`). Never on validation or test.

In [ ]:
def normalize_np(x, a, b):
    if NORMALIZE == "minmax":
        return np.clip((x - a) / (b - a), 0.0, 1.0)      # a = lo, b = hi
    return (x - a) / b                                   # a = mean, b = std (one number per mel row)

def make_prepare(a, b, augment):
    a_t, b_t = tf.constant(a, tf.float32), tf.constant(b, tf.float32)
    def prepare(x):
        if augment:
            x = x + tf.random.uniform((tf.shape(x)[0], 1, 1), -GAIN_DB, GAIN_DB)    # random volume
        if NORMALIZE == "minmax":
            x = tf.clip_by_value((x - a_t) / (b_t - a_t), 0.0, 1.0)
        else:
            x = (x - a_t) / b_t
        x = x[..., None]
        return x, x                                                                 # the target is the input itself
    return prepare

def make_dataset(pictures, a, b, augment, shuffle):
    ds = tf.data.Dataset.from_tensor_slices(pictures)
    if shuffle:
        ds = ds.shuffle(min(len(pictures), 20000), seed=SEED)
    ds = ds.batch(BATCH).map(make_prepare(a, b, augment))
    return ds.prefetch(tf.data.AUTOTUNE)

## Step 10 - The model (CAE)

- **Encoder:** 4 blocks of Conv (stride 2) + BatchNorm + LeakyReLU: 32, 64, 128, 256 filters.
- **Middle:** a Dense layer with only 32 numbers.
- **Decoder:** 4 blocks of transposed Conv: 128, 64, 32, 16 filters, then a last Conv (sigmoid for `minmax`, linear for `zscore`).
- No skip connections, so the model cannot copy faults around the middle layer.

In [ ]:
def build_cae():
    inputs = tf.keras.Input(shape=(N_MELS, FRAMES, 1))
    x = inputs
    for filters in (32, 64, 128, 256):
        x = tf.keras.layers.Conv2D(filters, 3, strides=2, padding="same")(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.LeakyReLU(alpha=0.2)(x)
    h, w = x.shape[1], x.shape[2]                       # 8 x 4 after 4 halvings
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(LATENT, name="bottleneck")(x)
    x = tf.keras.layers.Dense(h * w * 256)(x)
    x = tf.keras.layers.LeakyReLU(alpha=0.2)(x)
    x = tf.keras.layers.Reshape((h, w, 256))(x)
    for filters in (128, 64, 32, 16):
        x = tf.keras.layers.Conv2DTranspose(filters, 3, strides=2, padding="same")(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.LeakyReLU(alpha=0.2)(x)
    outputs = tf.keras.layers.Conv2D(1, 3, padding="same", activation="sigmoid" if NORMALIZE == "minmax" else None)(x)
    return tf.keras.Model(inputs, outputs)

build_cae().summary()

## Step 11 - Train the model

Early stopping (patience 10) and learning-rate reduction watch the validation loss.

In [ ]:
def train_model(X_train, X_val, a, b):
    model = build_cae()
    model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE), loss="mse")
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
    ]
    history = model.fit(make_dataset(X_train, a, b, AUGMENT, shuffle=True),
                        validation_data=make_dataset(X_val, a, b, False, shuffle=False),
                        epochs=EPOCHS, callbacks=callbacks, verbose=0)
    return model, history.history

## Step 12 - Score the clips

**Score = copy error (MSE).** Each window gets a score. A clip score is the **average** of its window scores.

In [ ]:
def scale(pictures, a, b):
    return normalize_np(pictures, a, b)[..., None].astype(np.float32)

def clip_scores(model, pictures, clip_id, a, b, chunk=2048):
    errors = []
    for i in range(0, len(pictures), chunk):
        x = scale(pictures[i:i + chunk], a, b)
        rebuilt = model.predict(x, batch_size=256, verbose=0)
        errors.append(np.mean((x - rebuilt) ** 2, axis=(1, 2, 3)))
    return pd.Series(np.concatenate(errors)).groupby(clip_id).mean().values

def get_example(model, pictures, clip_id, which_clip, a, b):
    x = scale(pictures[clip_id == which_clip][:1], a, b)
    rebuilt = model.predict(x, verbose=0)[0, ..., 0]
    return {"original": x[0, ..., 0], "rebuilt": rebuilt, "error": np.abs(x[0, ..., 0] - rebuilt)}

## Step 13 - Measure the result

**Threshold** = 95th percentile of healthy **validation** scores. About 5% of healthy clips raise a false alarm.

Numbers we report: AUC, pAUC, F1, **recall**, **false alarm**, **recall at 5% false alarm** (read from the ROC curve), **accuracy** and **balanced accuracy**.

In [ ]:
def get_metrics(val_scores, test_scores, y):
    threshold = np.percentile(val_scores, THRESHOLD_PERCENT)
    pred = (test_scores > threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    fpr, tpr, _ = roc_curve(y, test_scores)
    return {
        "auc":  roc_auc_score(y, test_scores),
        "pauc": roc_auc_score(y, test_scores, max_fpr=0.1),
        "f1":   f1_score(y, pred),
        "accuracy": (tp + tn) / len(y),
        "bal_acc": 0.5 * (tp / (tp + fn) + tn / (tn + fp)),
        "recall": tp / (tp + fn),
        "false_alarm": fp / (fp + tn),
        "recall_at_5pct_fa": float(np.interp(0.05, fpr, tpr)),
        "threshold": threshold,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

In [ ]:
# Published AUC from the MIMII paper (Table 3), for comparison.
# Order inside each tuple: 6 dB, 0 dB, -6 dB
PUBLISHED = {
 "fan":    {"id_00": (.75,.63,.57), "id_01": (.97,.90,.70), "id_02": (.99,.83,.68), "id_03": (1.0,.89,.70),
            "id_04": (.92,.75,.57), "id_05": (.95,.90,.83), "id_06": (.99,.97,.83)},
 "pump":   {"id_00": (.84,.65,.58), "id_01": (.98,.90,.73), "id_02": (.45,.46,.52), "id_03": (.79,.81,.75),
            "id_04": (.99,.95,.93), "id_05": (.66,.66,.64), "id_06": (.94,.76,.61)},
 "slider": {"id_00": (.99,.99,.93), "id_01": (.94,.90,.83), "id_02": (.93,.79,.74), "id_03": (.99,.85,.71),
            "id_04": (.88,.78,.61), "id_05": (.84,.70,.60), "id_06": (.71,.56,.52)},
 "valve":  {"id_00": (.68,.55,.62), "id_01": (.77,.71,.61), "id_02": (.66,.59,.57), "id_03": (.70,.65,.44),
            "id_04": (.64,.65,.50), "id_05": (.52,.48,.44), "id_06": (.70,.66,.53)},
}
SNR_COLUMN = {6: 0, 0: 1, -6: 2}

## Step 14 - Save files for later

After each run we save into `/kaggle/working/outputs_cae/<machine>_<id>/`:

| File | What it holds |
|---|---|
| `model.keras` | the trained model (weights + layout) |
| `norm.npz` | the scaling numbers (needed to reuse the model) |
| `clip_scores.csv` | one row per clip: file, label, score, alarm yes/no |
| `history.csv` | loss per epoch |
| `metrics.json` | AUC, pAUC, F1, threshold, ... |

In [ ]:
OUT_DIR = WORK_DIR + "/outputs_" + TAG

def save_run(machine, mid, model, norm, val_files, test_files, saved):
    folder = f"{OUT_DIR}/{machine}_{mid}"
    os.makedirs(folder, exist_ok=True)
    model.save(folder + "/model.keras")
    np.savez(folder + "/norm.npz", **norm)
    thr = saved["metrics"]["threshold"]
    test_table = pd.DataFrame({"machine": machine, "id": mid, "snr": SNR, "split": "test",
        "file": [str(f) for f in test_files], "label": saved["y_test"],
        "score": saved["test_scores"], "alarm": (saved["test_scores"] > thr).astype(int)})
    val_table = pd.DataFrame({"machine": machine, "id": mid, "snr": SNR, "split": "val",
        "file": [str(f) for f in val_files], "label": 0,
        "score": saved["val_scores"], "alarm": (saved["val_scores"] > thr).astype(int)})
    pd.concat([val_table, test_table]).to_csv(folder + "/clip_scores.csv", index=False)
    pd.DataFrame(saved["history"]).to_csv(folder + "/history.csv", index_label="epoch")
    with open(folder + "/metrics.json", "w") as f:
        json.dump({k: float(v) for k, v in saved["metrics"].items()}, f, indent=2)

## Step 15 - One full run

Load, split, train, score, measure, save. Healthy clips only are used for training.

In [ ]:
def run_one(machine, mid):
    t0 = time.time()
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(SEED)

    healthy, faulty = get_files(machine, mid)
    train_f, val_f, test_healthy_f = split_healthy(healthy)
    test_f = test_healthy_f + faulty
    y_test = np.array([0] * len(test_healthy_f) + [1] * len(faulty))     # 1 = faulty
    print(f"  clips: train {len(train_f)}, val {len(val_f)}, test {len(test_healthy_f)} healthy + {len(faulty)} faulty")

    X_train, _           = load_clips(train_f)
    X_val,  val_clip_id  = load_clips(val_f)
    X_test, test_clip_id = load_clips(test_f)

    if NORMALIZE == "minmax":                                         # scaling numbers from the train set only
        a, b = float(X_train.min()), float(X_train.max())
    else:
        a = X_train.mean(axis=(0, 2), keepdims=True)
        b = X_train.std(axis=(0, 2), keepdims=True) + 1e-6
    model, history = train_model(X_train, X_val, a, b)

    val_scores  = clip_scores(model, X_val,  val_clip_id,  a, b)
    test_scores = clip_scores(model, X_test, test_clip_id, a, b)
    metrics = get_metrics(val_scores, test_scores, y_test)
    examples = {"healthy": get_example(model, X_test, test_clip_id, 0, a, b),
                "faulty":  get_example(model, X_test, test_clip_id, len(test_healthy_f), a, b)}

    published = PUBLISHED[machine][mid][SNR_COLUMN[SNR]] if SNR in SNR_COLUMN else np.nan
    row = {"machine": machine, "id": mid, "snr": SNR, **metrics, "published_auc": published,
           "epochs": len(history["loss"]), "minutes": round((time.time() - t0) / 60, 1)}
    saved = {"history": history, "val_scores": val_scores, "test_scores": test_scores, "y_test": y_test,
             "metrics": metrics, "examples": examples}
    save_run(machine, mid, model, {"a": np.array(a), "b": np.array(b)}, val_f, test_f, saved)
    return row, saved

## Step 16 - Run everything

This is the long step. `results_cae.csv` is saved after **every** run. If Kaggle stops, you keep the finished runs.

In [ ]:
results = []        # one row per run
SAVED = {}          # scores and loss curves for the charts
start = time.time()

for number, (machine, mid) in enumerate(plan, 1):
    if (time.time() - start) / 3600 > MAX_HOURS:
        print("Time limit reached. Stopping.")
        break
    print(f"[{number}/{len(plan)}] {machine} {mid}  ({(time.time() - start) / 60:.1f} min so far)")
    row, saved = run_one(machine, mid)
    results.append(row)
    SAVED[(machine, mid)] = saved
    pd.DataFrame(results).to_csv(WORK_DIR + f"/results_{TAG}.csv", index=False)
    print(f"  AUC {row['auc']:.3f} | pAUC {row['pauc']:.3f} | recall {row['recall']:.3f} | false alarm {row['false_alarm']:.3f} | published AUC {row['published_auc']} | {row['epochs']} epochs | {row['minutes']} min")

head = pd.DataFrame(results)      # one row per run
print(f"Done in {(time.time() - start) / 60:.1f} minutes")

## Part A - Report for each machine

Each row of charts shows one machine (all its IDs):
1. **Loss curve** - solid = train, dashed = validation (one colour per ID). *(only for methods that train)*
2. **ROC curve** - one line per ID.
3. **Confusion matrix** - all IDs of this machine added together, at the threshold.
4. **Score distribution** - healthy against faulty test clips. Scores are divided by the threshold, so **1.0 = alarm line**.

In [ ]:
MACHINE_COLOR = {"fan": "#1f77b4", "pump": "#ff7f0e", "slider": "#2ca02c", "valve": "#d62728"}
FIG_DIR = globals().get("FIG_DIR", WORK_DIR + "/figures")
os.makedirs(FIG_DIR, exist_ok=True)
HAS_LOSS = True

def show(name):
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/{name}.png", dpi=110, bbox_inches="tight")
    plt.show()

id_colors = dict(zip(sorted(set(k[1] for k in SAVED)), plt.cm.tab10.colors))

def machine_report(machine):
    runs = [k for k in SAVED if k[0] == machine]
    if not runs:
        print("No runs for", machine); return
    n = 4 if HAS_LOSS else 3
    fig, axs = plt.subplots(1, n, figsize=(5.2 * n, 4.2))
    axs = list(np.atleast_1d(axs))
    cm = np.zeros((2, 2), int)
    h_all, f_all = [], []
    for k in runs:
        r = SAVED[k]; m = r["metrics"]; c = id_colors[k[1]]; thr = m["threshold"]
        i = 0
        if HAS_LOSS:
            axs[0].plot(r["history"]["loss"], color=c, label=k[1])
            axs[0].plot(r["history"]["val_loss"], "--", color=c)
            i = 1
        fpr, tpr, _ = roc_curve(r["y_test"], r["test_scores"])
        axs[i].plot(fpr, tpr, color=c, label=f"{k[1]} (AUC {m['auc']:.2f})")
        cm += np.array([[m["tn"], m["fp"]], [m["fn"], m["tp"]]])
        h_all += list(r["test_scores"][r["y_test"] == 0] / thr)
        f_all += list(r["test_scores"][r["y_test"] == 1] / thr)
    i = 0
    if HAS_LOSS:
        axs[0].set(title=f"{machine}: loss (solid train, dashed validation)", xlabel="epoch", ylabel="loss", yscale="log"); axs[0].legend()
        i = 1
    axs[i].plot([0, 1], [0, 1], "k:", lw=1); axs[i].axvline(0.05, color="gray", ls=":", lw=1)
    axs[i].set(title=f"{machine}: ROC", xlabel="false alarm rate", ylabel="faults caught"); axs[i].legend(loc="lower right")
    ConfusionMatrixDisplay(cm, display_labels=["healthy", "faulty"]).plot(ax=axs[i + 1], cmap="Blues", colorbar=False)
    axs[i + 1].set_title(f"{machine}: confusion matrix (all IDs)")
    hi = np.percentile(h_all + f_all, 99)
    bins = np.linspace(0, hi, 40)
    axs[i + 2].hist(np.clip(h_all, 0, hi), bins=bins, alpha=0.6, color="tab:green", label="healthy (test)")
    axs[i + 2].hist(np.clip(f_all, 0, hi), bins=bins, alpha=0.6, color="tab:red", label="faulty (test)")
    axs[i + 2].axvline(1.0, color="k", ls="--", label="alarm line")
    axs[i + 2].set(title=f"{machine}: scores (divided by each run's threshold)", xlabel="score / threshold"); axs[i + 2].legend()
    show(f"machine_{machine}")
    tcols = [c for c in ["id", "auc", "pauc", "accuracy", "bal_acc", "recall", "false_alarm", "recall_at_5pct_fa", "epochs"] if c in head.columns]
    display(head[head.machine == machine][tcols].round(3).reset_index(drop=True))

### Fan

In [ ]:
machine_report("fan")

### Pump

In [ ]:
machine_report("pump")

### Slider

In [ ]:
machine_report("slider")

### Valve

In [ ]:
machine_report("valve")

# Part B - Report for all machines together

The same charts, with more detail, for **all runs**.

### B1 - Summary table (every run)

In [ ]:
tcols = [c for c in ["machine", "id", "auc", "pauc", "f1", "accuracy", "bal_acc", "recall", "false_alarm", "recall_at_5pct_fa", "epochs"] if c in head.columns]
mcols = ["auc", "pauc", "accuracy", "bal_acc", "recall", "false_alarm", "recall_at_5pct_fa"]
display(head[tcols].round(3).reset_index(drop=True))
print("Mean per machine:")
display(head.groupby("machine")[mcols].mean().round(3))
print("Overall mean:")
display(head[mcols].mean().round(3).to_frame("mean").T)

### B2 - Confusion matrices

Left: **all runs added together**. Right: one matrix per machine, as **percent of each true class** (rows add to 100%).

In [ ]:
def matrix(df):
    return np.array([[df.tn.sum(), df.fp.sum()], [df.fn.sum(), df.tp.sum()]])

fig, axs = plt.subplots(1, 5, figsize=(22, 4.2))
ConfusionMatrixDisplay(matrix(head), display_labels=["healthy", "faulty"]).plot(ax=axs[0], cmap="Blues", colorbar=False)
axs[0].set_title("ALL machines (counts)")
for ax, machine in zip(axs[1:], MACHINES):
    sub = head[head.machine == machine]
    if len(sub) == 0:
        ax.axis("off"); continue
    cm = matrix(sub); pct = cm / cm.sum(axis=1, keepdims=True) * 100
    ConfusionMatrixDisplay(pct, display_labels=["healthy", "faulty"]).plot(ax=ax, cmap="Blues", colorbar=False, values_format=".0f")
    ax.set_title(f"{machine} (% of true class)")
show("confusion_all")

### B3 - Confusion matrix for every run

In [ ]:
ids = sorted(head["id"].unique())
fig, axs = plt.subplots(len(MACHINES), len(ids), figsize=(4 * len(ids), 3.6 * len(MACHINES)), squeeze=False)
for i, machine in enumerate(MACHINES):
    for j, mid in enumerate(ids):
        ax = axs[i, j]
        sub = head[(head.machine == machine) & (head["id"] == mid)]
        if len(sub) == 0:
            ax.axis("off"); continue
        r = sub.iloc[0]
        ConfusionMatrixDisplay(np.array([[r.tn, r.fp], [r.fn, r.tp]]), display_labels=["healthy", "faulty"]).plot(ax=ax, cmap="Blues", colorbar=False)
        ax.set_title(f"{machine} {mid}  (bal.acc {r.bal_acc:.2f})", fontsize=10)
show("confusion_every_run")

### B4 - Loss curves, all runs

Colour = machine. Solid = train, dashed = validation.

In [ ]:
if HAS_LOSS:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for (machine, mid), r in SAVED.items():
        c = MACHINE_COLOR.get(machine, "gray")
        ax.plot(r["history"]["loss"], color=c, alpha=0.6); ax.plot(r["history"]["val_loss"], "--", color=c, alpha=0.6)
    for machine in MACHINES:
        ax.plot([], [], color=MACHINE_COLOR[machine], label=machine)
    ax.set(title="Loss - all runs", xlabel="epoch", ylabel="loss", yscale="log"); ax.legend()
    show("loss_all")
else:
    print("This method has no training, so there is no loss curve.")

### B5 - ROC curves

Thin lines = single runs. The legend shows the **mean AUC** of each machine.
(We do not pool scores of different runs: their scales are different.)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))
for machine in MACHINES:
    aucs = []
    for (mc, mid), r in SAVED.items():
        if mc != machine: continue
        fpr, tpr, _ = roc_curve(r["y_test"], r["test_scores"])
        ax.plot(fpr, tpr, color=MACHINE_COLOR[machine], alpha=0.55, lw=1.3)
        aucs.append(r["metrics"]["auc"])
    if aucs:
        ax.plot([], [], color=MACHINE_COLOR[machine], lw=2, label=f"{machine} (mean AUC {np.mean(aucs):.2f})")
ax.plot([0, 1], [0, 1], "k:", lw=1); ax.axvline(0.05, color="gray", ls=":", lw=1)
ax.set(xlabel="false alarm rate", ylabel="faults caught", title="ROC - all runs"); ax.legend(loc="lower right")
show("roc_all")

### B6 - Score distributions per machine

Scores are divided by each run's threshold, so **1.0 = the alarm line**.

In [ ]:
fig, axs = plt.subplots(1, len(MACHINES), figsize=(5 * len(MACHINES), 3.8))
for ax, machine in zip(np.atleast_1d(axs), MACHINES):
    h, f = [], []
    for (mc, mid), r in SAVED.items():
        if mc != machine: continue
        thr = r["metrics"]["threshold"]
        h += list(r["test_scores"][r["y_test"] == 0] / thr); f += list(r["test_scores"][r["y_test"] == 1] / thr)
    if not h:
        ax.axis("off"); continue
    hi = np.percentile(h + f, 99); bins = np.linspace(0, hi, 40)
    ax.hist(np.clip(h, 0, hi), bins=bins, alpha=0.6, color="tab:green", label="healthy"); ax.hist(np.clip(f, 0, hi), bins=bins, alpha=0.6, color="tab:red", label="faulty")
    ax.axvline(1.0, color="k", ls="--"); ax.set(title=machine, xlabel="score / threshold"); ax.legend()
show("scores_per_machine")

### B7 - Heatmaps (machine x ID)

In [ ]:
def heat(ax, table, title, vmin, vmax):
    im = ax.imshow(table.values, vmin=vmin, vmax=vmax, cmap="viridis")
    ax.set(xticks=range(table.shape[1]), xticklabels=table.columns, yticks=range(table.shape[0]), yticklabels=table.index, title=title)
    for i in range(table.shape[0]):
        for j in range(table.shape[1]):
            v = table.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="white" if v < (vmin + vmax) / 2 else "black")
    plt.colorbar(im, ax=ax, fraction=0.046)

fig, axs = plt.subplots(1, 3, figsize=(17, 3.8))
heat(axs[0], head.pivot(index="machine", columns="id", values="auc"), "AUC", 0.5, 1.0)
heat(axs[1], head.pivot(index="machine", columns="id", values="bal_acc"), "Balanced accuracy", 0.5, 1.0)
heat(axs[2], head.pivot(index="machine", columns="id", values="recall_at_5pct_fa"), "Faults caught at 5% false alarm", 0.0, 1.0)
show("heatmaps")

### B8 - Counts of mistakes

Missed faults and false alarms, summed over the IDs of each machine.

In [ ]:
mist = head.groupby("machine")[["fn", "fp", "tp", "tn"]].sum()
mist["missed_faults_%"] = (100 * mist.fn / (mist.fn + mist.tp)).round(1)
mist["false_alarms_%"] = (100 * mist.fp / (mist.fp + mist.tn)).round(1)
display(mist)
ax = mist[["missed_faults_%", "false_alarms_%"]].plot(kind="bar", figsize=(7, 4), rot=0, color=["tab:red", "tab:orange"])
ax.set(ylabel="percent", title="Mistakes per machine")
show("mistakes")

### B9 - Ours vs the published result

The published model is the MIMII paper's own healthy-only autoencoder. Same idea as ours, so this is a fair reference line, not an exact match (different split).

In [ ]:
g = head.groupby("machine")[["auc", "published_auc"]].mean()
ax = g.plot(kind="bar", figsize=(7, 4), rot=0)
ax.axhline(0.5, color="gray", ls="--", lw=1); ax.set(ylabel="mean AUC", ylim=(0.3, 1.02), title="Ours vs published")
ax.legend(["ours", "published"])
show("vs_published")

## Part C - Original vs rebuilt (CAE only)

For one run: a healthy and a faulty test window. **Original**, **rebuilt by the model** and the **absolute error**.
A faulty window should show more error. Change `KEY` to look at another run.

In [ ]:
KEY = ("valve", "id_00")          # <-- change this
if KEY not in SAVED:
    KEY = list(SAVED)[0]
    print("That run was not found. Using", KEY)
ex = SAVED[KEY]["examples"]
fig, axs = plt.subplots(2, 3, figsize=(14, 6))
for row, kind in enumerate(["healthy", "faulty"]):
    for col, part in enumerate(["original", "rebuilt", "error"]):
        a = axs[row, col]
        fixed = (NORMALIZE == "minmax") and part != "error"
        im = a.imshow(ex[kind][part], origin="lower", aspect="auto", cmap="magma" if part != "error" else "viridis",
                      vmin=0 if fixed else None, vmax=1 if fixed else None)
        a.set(title=f"{kind} - {part}", xlabel="time", ylabel="mel band"); plt.colorbar(im, ax=a, fraction=0.046)
    print(f"{kind}: mean error {ex[kind]['error'].mean():.4f}")
show("reconstruction_examples")

## Save files

- `results_cae.csv` = one row per run.
- `all_clip_scores_cae.csv` = every clip of every run.
- `history_cae.csv` = loss per epoch for every run.
- `figures_cae/` = every chart as a PNG.
- `export_cae.zip` = all of the above, easy to download from the Output tab.

In [ ]:
parts = [pd.read_csv(f) for f in sorted(glob.glob(OUT_DIR + "/*/clip_scores.csv"))]
all_scores = pd.concat(parts, ignore_index=True)
hist_parts = []
for f in sorted(glob.glob(OUT_DIR + "/*/history.csv")):
    h = pd.read_csv(f); h.insert(0, "run", f.split("/")[-2]); hist_parts.append(h)

EXP = WORK_DIR + "/export_" + TAG
os.makedirs(EXP, exist_ok=True)
head.to_csv(EXP + f"/results_{TAG}.csv", index=False)
all_scores.to_csv(EXP + f"/all_clip_scores_{TAG}.csv", index=False)
all_scores.to_csv(WORK_DIR + f"/all_clip_scores_{TAG}.csv", index=False)
pd.concat(hist_parts, ignore_index=True).to_csv(EXP + f"/history_{TAG}.csv", index=False)
shutil.copytree(FIG_DIR, EXP + "/figures_" + TAG, dirs_exist_ok=True)
shutil.make_archive(WORK_DIR + "/export_" + TAG, "zip", EXP)
print("all_clip_scores:", all_scores.shape, "| saved:", os.listdir(EXP), "-> /kaggle/working/export_" + TAG + ".zip")

## Summary to paste back

Run this cell and copy its output.

In [ ]:
print("MODEL:", TAG, "| QUICK:", QUICK, "| SPLIT_MODE:", SPLIT_MODE, "| IDS:", IDS, "| LIMIT:", LIMIT)
print("=== EVERY RUN ===")
print(head[[c for c in ["machine", "id", "auc", "pauc", "f1", "accuracy", "bal_acc", "recall", "false_alarm", "recall_at_5pct_fa", "published_auc", "epochs", "minutes"] if c in head.columns]].round(3).to_csv(index=False))
print("=== MEAN PER MACHINE ===")
print(head.groupby("machine")[["auc", "pauc", "accuracy", "bal_acc", "recall", "false_alarm", "recall_at_5pct_fa", "published_auc"]].mean().round(3).to_csv())
print("=== OVERALL MEAN ===")
print(head[["auc", "pauc", "accuracy", "bal_acc", "recall", "false_alarm", "recall_at_5pct_fa", "published_auc"]].mean().round(3).to_csv())
print("=== MISTAKE COUNTS ===")
print(mist.to_csv())